# **Интерпретация оценки CATE**

Практика к модулю [«Гетерогенные эффекты»](https://ai-interpretability.school), урок «Интерпретация оценки CATE».

Оценка $\hat\tau(x)$ — сама модель, и ее тоже надо объяснять: по каким признакам различается эффект, где проходят границы подгрупп, насколько уверенно. В уроке разобраны четыре способа: важность признаков причинного леса, линейная проекция дважды робастных оценок на признаки (BLP по признакам), значения Шепли для модели эффекта и дерево-интерпретатор. Здесь мы применим их к одной и той же оценке и проверим три утверждения урока:

1. важность признака для исхода и важность для эффекта — разные величины;
2. четыре способа отвечают на разные вопросы и расходятся, когда признаки коррелируют;
3. объяснение модели эффекта не проверяет, есть ли неоднородность вообще: SHAP и дерево-интерпретатор раскладывают и шум.

План тетради:

1. синтетические данные с известным $\tau(x)$ и причинный лес;
2. важность для исхода против важности для эффекта;
3. BLP по признакам;
4. значения Шепли для модели эффекта;
5. дерево-интерпретатор;
6. где ломается: коррелирующие признаки и эффект без неоднородности;
7. применение: программа growth mindset в школах;
8. реализации.

Тетрадь идет на процессоре; весь прогон занимает две-три минуты.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import logging
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # shap при импорте ищет виджеты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import shap
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
from econml.dml import CausalForestDML
from econml.cate_interpreter import SingleTreeCateInterpreter

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # ложные предупреждения numpy на некоторых процессорах
logging.getLogger('shap').setLevel(logging.ERROR)                     # shap сообщает в журнал о подвыборке фона
SEED = 0
pd.set_option('display.width', 140)

## 1. Данные и причинный лес

Рандомизированный эксперимент на 4000 объектах, шесть признаков $X\sim\mathcal N(0, I)$, кроме $x_4$: он коррелирует с $x_0$ (коэффициент корреляции $0.9$). Воздействие получает случайная половина.

- Эффект: $\tau(x)=1+1.5\cdot\mathbb 1[x_0>0]+0.5\,x_3$ — ступенька по $x_0$ и наклон по $x_3$.
- Исход без воздействия: $\mu_0(x)=3x_1+2x_2^2+x_3$. Признаки $x_1$ и $x_2$ сильно влияют на исход и никак — на эффект; $x_3$ влияет на оба.
- $x_4$ на исход и эффект не влияет, но похож на $x_0$; $x_5$ — чистый шум.
- $Y=\mu_0(X)+T\,\tau(X)+\varepsilon$, $\varepsilon\sim\mathcal N(0,1)$.

Половина объектов идет на обучение леса, половина — на проверку и на все объяснения.

In [ ]:
def simulate(n, r, heterogeneous=True):
    Xs = r.normal(size=(n, 6))
    Xs[:, 4] = 0.9 * Xs[:, 0] + np.sqrt(1 - 0.9 ** 2) * r.normal(size=n)   # x4 похож на x0
    ts = 1 + 1.5 * (Xs[:, 0] > 0) + 0.5 * Xs[:, 3] if heterogeneous else np.ones(n)
    Ts = r.binomial(1, 0.5, n)
    Ys = 3 * Xs[:, 1] + 2 * Xs[:, 2] ** 2 + Xs[:, 3] + Ts * ts + r.normal(size=n)
    return Xs, Ts, Ys, ts


def gb():
    return GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=SEED)


def forest(Xa, Ta, Ya):
    return CausalForestDML(model_y=gb(), model_t=LogisticRegression(), discrete_treatment=True,
                           n_estimators=500, min_samples_leaf=20, random_state=SEED).fit(Ya, Ta, X=Xa)


names = [f'x{i}' for i in range(6)]
X, T, Y, tau = simulate(4000, np.random.default_rng(SEED))
tr, te = np.arange(2000), np.arange(2000, 4000)
cf = forest(X[tr], T[tr], Y[tr])
tau_hat = cf.effect(X[te])
print('RMSE оценки CATE на проверочной половине:', round(np.sqrt(np.mean((tau_hat - tau[te]) ** 2)), 3))
print('корреляция x0 и x4:', round(np.corrcoef(X[:, 0], X[:, 4])[0, 1], 3))

## 2. Важность для исхода против важности для эффекта

Первая ловушка урока «Как читать оценку и где ошибиться»: признак, важный для прогноза исхода, не обязан различать эффект. Сравним две важности на одних и тех же данных.

- **Важность для исхода** — permutation importance модели исхода: бустинг на признаках и воздействии, падение $R^2$ на проверочной половине при перемешивании признака.
- **Важность для эффекта** — `feature_importances_` причинного леса. В EconML это не частота расщеплений: каждое расщепление по признаку добавляет к его важности то, насколько оно разводит оценки эффекта в двух дочерних узлах, с весом $1/(1+\text{глубина})^2$ и только до глубины 4; затем важности нормируются на сумму 1.

**Задание 1. Обучите модель исхода `gb()` на `np.c_[X[tr], T[tr]]` и `Y[tr]` и посчитайте `permutation_importance` на проверочной половине (`n_repeats=5`, `random_state=SEED`). Выведите таблицу из двух столбцов: важность для исхода и `cf.feature_importances_`. Какая доля нормированной важности леса приходится на $x_0$?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 3. BLP по признакам

Важность леса — число без знака и без интервала. Линейная проекция дает и то и другое. Для каждого проверочного объекта считаем дважды робастную оценку эффекта

$$\hat\Gamma_i=\hat\mu_1(X_i)-\hat\mu_0(X_i)+\frac{T_i\,(Y_i-\hat\mu_1(X_i))}{e}-\frac{(1-T_i)\,(Y_i-\hat\mu_0(X_i))}{1-e}$$

с моделями исхода, обученными на других фолдах проверочной половины; склонность в эксперименте известна, $e=0.5$. Затем регрессируем $\hat\Gamma_i$ на признаки методом наименьших квадратов: $\hat\Gamma_i\approx\beta_0+\sum_j\beta_jX_{ij}$. Коэффициент $\beta_j$ — наклон лучшего линейного приближения к $\tau(x)$ по признаку $x_j$ при остальных фиксированных, а стандартная ошибка в устойчивой к гетероскедастичности форме (HC3) дает интервал. Это BLP по признакам ([Semenova и Chernozhukov, 2017](https://arxiv.org/abs/1702.06240)); в R-пакете `grf` — функция `best_linear_projection()`.

Обратите внимание: проекция описывает $\tau(x)$, а не модель леса. Лес нужен только для сравнения с другими способами.

In [ ]:
def aipw_scores(Xa, Ta, Ya, e=0.5):
    # дважды робастные оценки с cross-fitting; склонность известна
    G = np.zeros(len(Ya))
    for a, b in KFold(5, shuffle=True, random_state=SEED).split(Xa):
        m1 = gb().fit(Xa[a][Ta[a] == 1], Ya[a][Ta[a] == 1]).predict(Xa[b])
        m0 = gb().fit(Xa[a][Ta[a] == 0], Ya[a][Ta[a] == 0]).predict(Xa[b])
        G[b] = m1 - m0 + Ta[b] * (Ya[b] - m1) / e - (1 - Ta[b]) * (Ya[b] - m0) / (1 - e)
    return G


G_te = aipw_scores(X[te], T[te], Y[te])
print('среднее оценок Gamma (оценка ATE):', round(G_te.mean(), 3), '| истинный средний эффект:', round(tau[te].mean(), 3))

**Задание 2. Регрессируйте `G_te` на признаки проверочной половины с константой (`sm.OLS(..., sm.add_constant(...)).fit(cov_type='HC3')`), подпишите строки именами признаков и выведите коэффициенты, стандартные ошибки и p-значения. Чему равен коэффициент при $x_3$ и как он соотносится с истинным наклоном $0.5$? Как проекция делит ступеньку по $x_0$ между $x_0$ и похожим на него $x_4$?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 4. Значения Шепли для модели эффекта

SHAP раскладывает прогноз модели $\hat\tau(x)$ одного объекта на вклады признаков относительно среднего прогноза по фону: $\hat\tau(x)=\phi_0+\sum_j\phi_j(x)$. Это то же разложение, что в уроке «Основы семейства SHAP-методов» модуля о классических методах, только модель — оценка эффекта, а не прогноз исхода. Вклад $\phi_j(x)$ отвечает на вопрос «насколько признак $x_j$ этого объекта сдвигает *оценку эффекта*», и ничего не говорит о том, верна ли сама оценка.

`cf.shap_values(X)` в EconML строит `shap.Explainer` над моделью эффекта с маской `shap.maskers.Independent` и фоном из 100 объектов, взятых из переданных `X`: значения интервенционные, признаки при маскировании берутся из фона независимо друг от друга.

**Задание 3. Посчитайте `cf.shap_values(X[te][:300], feature_names=names)` и достаньте объект `shap.Explanation` (словарь словарей: по исходу, затем по воздействию). Выведите средний модуль вклада $\overline{|\phi_j|}$ по 300 объектам для каждого признака и нарисуйте `shap.plots.beeswarm`. Какой средний модуль вклада у $x_0$ и у $x_4$?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 5. Дерево-интерпретатор

`SingleTreeCateInterpreter` обучает обычное дерево регрессии `sklearn` предсказывать оценку $\hat\tau(x)$ леса по признакам — то есть строит суррогат модели эффекта — ту же идею, что у суррогатной модели из урока про LIME модуля о классических методах, только суррогат один на всю выборку. Листья дерева — подгруппы, которые легко назвать словами; число в листе — средняя оценка эффекта леса, а не средний фактический эффект. Дерево объясняет модель, а не данные: его разбиения нельзя читать как проверенные подгруппы.

**Задание 4. Постройте `SingleTreeCateInterpreter(max_depth=2, min_samples_leaf=100)` по `cf` и `X[te]`, нарисуйте его методом `plot(feature_names=names)` и выведите признак и порог первого разбиения из `tree_model_.tree_`. По какому признаку и при каком пороге проходит первое разбиение?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 6. Где ломается

### 6.1. Коррелирующие признаки

Сведем в одну таблицу то, что четыре способа говорят о $x_0$ и $x_4$. Признак $x_4$ на эффект не влияет, но несет почти ту же информацию, что $x_0$.

In [ ]:
summary = pd.DataFrame({'важность леса': imp.iloc[:, 1],
                        'BLP: коэффициент': blp.params[names],
                        'BLP: p-значение': blp.pvalues[names],
                        'SHAP: средний модуль': mean_abs}).loc[['x0', 'x4', 'x3', 'x1']]
print(summary.round(3))

Все три способа дают $x_4$ малое, но ненулевое значение, хотя на эффект он не влияет: лес иногда режет по $x_4$, когда в случайном подмножестве признаков узла нет $x_0$, а SHAP с независимой маской повторяет то, что сделала модель. Отличить $x_4$ от причины может только интервал: у коэффициента BLP при $x_4$ он накрывает ноль. Но и BLP здесь помогает лишь потому, что $x_0$ стоит в регрессии рядом: уберите $x_0$ из признаков, и проекция отдаст ступеньку $x_4$. Ни один способ не говорит «$x_4$ здесь не причина»: это знание о данных, а не о модели.

Вторая строка той же таблицы — масштаб. У $x_3$ важность леса в пять раз меньше, чем у $x_0$, а коэффициент BLP почти такой же: важность леса — доля неоднородности, созданной расщеплениями, наклон BLP — изменение эффекта на единицу признака. Сравнивать признаки по важности и по наклону — значит сравнивать разные величины.

### 6.2. Эффект без неоднородности

Теперь эффект у всех одинаковый, $\tau(x)=1$. Неоднородности нет, и любая структура в $\hat\tau(x)$ — шум оценки.

**Задание 5. Сгенерируйте данные `simulate(4000, np.random.default_rng(1), heterogeneous=False)`, обучите на первой половине лес функцией `forest` и повторите на второй половине три объяснения: важность леса, средний модуль SHAP на первых 300 объектах и BLP по признакам. Выведите сумму средних модулей SHAP по всем признакам, наибольшую важность леса и p-значение совместной проверки гипотезы «все коэффициенты при признаках равны нулю» (`f_test` или `wald_test` по шести коэффициентам; удобно `blp0.f_pvalue`). Что из трех объяснений сообщает, что неоднородности нет?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 7. Применение: программа growth mindset в школах

Данные те же, что в тетради «Причинный лес и честное расщепление»: симуляция по National Study of Learning Mindsets, подготовленная к семинару ACIC 2018 и разобранная причинным лесом у [Athey и Wager (2019)](https://arxiv.org/abs/1902.07409). 10391 ученица и ученик из 76 школ; воздействие `intervention` — короткая онлайн-программа о том, что способности развиваются; исход `achievement_score` — стандартизованная успеваемость. Воздействие назначено не случайно, поэтому здесь модели исхода и склонности в `CausalForestDML` обязательны; склонность оцениваем, а не знаем.

In [ ]:
URL = 'https://raw.githubusercontent.com/matheusfacure/python-causality-handbook/master/causal-inference-for-the-brave-and-true/data/learning_mindset.csv'
lm = pd.read_csv(URL)
Xm = pd.get_dummies(lm.drop(columns=['schoolid', 'intervention', 'achievement_score']),
                    columns=['ethnicity', 'gender', 'school_urbanicity'], drop_first=True).astype(float)
Tm, Ym = lm.intervention.values, lm.achievement_score.values
cfm = CausalForestDML(model_y=gb(), model_t=LogisticRegression(max_iter=1000), discrete_treatment=True,
                      n_estimators=500, min_samples_leaf=50, random_state=SEED).fit(Ym, Tm, X=Xm.values)
cate_m = cfm.effect(Xm.values)
print('учеников:', len(lm), '| признаков:', Xm.shape[1], '| средняя оценка CATE:', round(cate_m.mean(), 3))

**Задание 6. Для этой оценки выведите пять признаков с наибольшей важностью леса и пять признаков с наибольшим средним модулем SHAP (по 500 первым ученикам, `cfm.shap_values(Xm.values[:500], feature_names=list(Xm.columns))`), а затем постройте дерево-интерпретатор глубины 2 с `min_samples_leaf=500` и выведите признак и порог первого разбиения. Совпадает ли первый признак у трех объяснений?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 8. Реализации

- `econml` (лицензия MIT): `feature_importances_` у `CausalForestDML` и `econml.grf.CausalForest` — неоднородность, созданная расщеплениями, с весом по глубине; `shap_values()` у оценщиков CATE — `shap.Explainer` с независимой маской и фоном из 100 объектов; `econml.cate_interpreter.SingleTreeCateInterpreter` — дерево `sklearn` по прогнозу `const_marginal_effect`, с `include_model_uncertainty=True` еще и интервалы в листьях.
- R-пакет [grf](https://github.com/grf-labs/grf) (лицензия GPL-3.0): `variable_importance()` — взвешенная по глубине частота расщеплений, `best_linear_projection()` — BLP по признакам на дважды робастных оценках леса.
- [shap](https://github.com/shap/shap) (лицензия MIT): любые графики `shap.plots` для объекта `Explanation`, который возвращает EconML.

## Что унести из тетради

- **Важность для исхода и для эффекта — разные числа.** Признаки, которые сильнее всего объясняют исход, могут ничего не говорить об эффекте, и наоборот.
- **Четыре способа отвечают на разные вопросы.** Важность леса — где лес резал; BLP — линейный наклон эффекта с интервалом; SHAP — разложение оценки одного объекта; дерево-интерпретатор — подгруппы, которые легко назвать.
- **Коррелирующие признаки делят объяснение между собой,** и ни один способ не отличает причину от похожего на нее признака.
- **Объяснение не проверяет неоднородность.** SHAP, важность леса и дерево раскладывают и шум; проверку дает интервал BLP и проверки урока «Как читать оценку и где ошибиться».

Источники:
1. [Wager & Athey, 2018. Estimation and inference of heterogeneous treatment effects using random forests](https://arxiv.org/abs/1510.04342)
2. [Semenova & Chernozhukov, 2017. Debiased machine learning of conditional average treatment effects and other causal functions](https://arxiv.org/abs/1702.06240)
3. [Lundberg & Lee, 2017. A unified approach to interpreting model predictions](https://arxiv.org/abs/1705.07874)
4. [Athey & Wager, 2019. Estimating treatment effects with causal forests: an application](https://arxiv.org/abs/1902.07409)
5. [EconML](https://github.com/py-why/EconML), [shap](https://github.com/shap/shap)